# Homework 04: data acquisition and ingestion

**Name:** Paritosh Dwivedi  
**Date:** August 18, 2026

## Objective

This notebook acquires two timestamped raw-data snapshots. The API portion uses SPY daily closes so the exercise feeds the same market-data domain as the semester's Weekly ETF Risk Monitor. The scrape portion practices a separate permitted-table ingestion workflow using the current S&P 500 constituent list.


In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/paritoshdwivedi/Downloads/project bootcamp/bootcamp_paritosh_dwivedi/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import datetime as dt
import os
import pathlib
import subprocess

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
load_dotenv()
print("ALPHAVANTAGE_API_KEY loaded?", bool(os.getenv("ALPHAVANTAGE_API_KEY")))


ALPHAVANTAGE_API_KEY loaded? False


## Helpers


In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1: API pull

The requested instrument is SPY because its daily close history is the direct raw input domain for the Weekly ETF Risk Monitor. The starter's credential check selects Alpha Vantage when `ALPHAVANTAGE_API_KEY` is present and otherwise uses the course-sanctioned yfinance fallback.


In [5]:
SYMBOL = 'SPY'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

# Normalize both provider branches to the same typed, ordered raw schema.
df_api["date"] = pd.to_datetime(df_api["date"], errors="raise").dt.tz_localize(None)
df_api["close"] = pd.to_numeric(df_api["close"], errors="raise").astype("float64")
df_api = df_api.sort_values("date").reset_index(drop=True)

api_checks = {
    "rows_present": not df_api.empty,
    "dates_parsed": pd.api.types.is_datetime64_any_dtype(df_api["date"]),
    "closes_are_float": pd.api.types.is_float_dtype(df_api["close"]),
    "dates_unique": not df_api["date"].duplicated().any(),
    "dates_ascending": df_api["date"].is_monotonic_increasing,
    "closes_positive": df_api["close"].gt(0).all(),
}
assert all(api_checks.values()), api_checks
print("API source used:", "Alpha Vantage" if USE_ALPHA else "yfinance")
print("Basic rules:", api_checks)
print("Dtypes:", {column: str(dtype) for column, dtype in df_api.dtypes.items()})

v_api = validate(df_api, ['date','close']); v_api


[*********************100%***********************]  1 of 1 completed

API source used: yfinance
Basic rules: {'rows_present': True, 'dates_parsed': True, 'closes_are_float': True, 'dates_unique': True, 'dates_ascending': True, 'closes_positive': np.True_}
Dtypes: {'date': 'datetime64[ns]', 'close': 'float64'}


{'missing': [], 'shape': (63, 2), 'na_total': 0}

### Acquisition path used

The committed execution used yfinance because `ALPHAVANTAGE_API_KEY` was intentionally blank in the local `.env`. This is the fallback defined by the official starter and sanctioned by the homework sheet. The resulting SPY snapshot is suitable as a transparent raw input for later return and realized-volatility calculations.


In [6]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)


Saved data/raw/api_source-yfinance_symbol-SPY_20260818-194017.csv


## Part 2: scrape a public table

The S&P 500 constituent page is publicly readable and its content is available under Wikipedia's CC BY-SA terms. Wikipedia permits ordinary user-agent reads through its robots policy. This notebook makes one request with the starter's descriptive `AFE-Homework/1.0` user agent and selects the first table carrying the `wikitable` class, rather than depending on its position among all page tables.


In [7]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent':'AFE-Homework/1.0'}
SCRAPE_USED_FALLBACK = False

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', class_='wikitable')
    if table is None:
        raise ValueError("No table with class 'wikitable' was found")
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    SCRAPE_USED_FALLBACK = True
    html = '''
    <table class="wikitable">
      <tr><th>Symbol</th><th>Security</th><th>GICS Sector</th><th>GICS Sub-Industry</th><th>Headquarters Location</th><th>Date added</th><th>CIK</th><th>Founded</th></tr>
      <tr><td>DEMO</td><td>Inline fallback record</td><td>Not applicable</td><td>Not applicable</td><td>Not applicable</td><td>2000-01-01</td><td>1</td><td>Not applicable</td></tr>
    </table>
    '''
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', class_='wikitable')

# Read only direct row cells so nested markup cannot introduce extra columns.
table_rows = table.find_all('tr')
header = [cell.get_text(' ', strip=True) for cell in table_rows[0].find_all('th', recursive=False)]
data = []
for row in table_rows[1:]:
    values = [cell.get_text(' ', strip=True) for cell in row.find_all(['th', 'td'], recursive=False)]
    if len(values) == len(header):
        data.append(values)
df_scrape = pd.DataFrame(data, columns=header)

text_columns = [column for column in df_scrape.columns if column not in {'Date added', 'CIK'}]
for column in text_columns:
    df_scrape[column] = df_scrape[column].astype('string').str.strip()
df_scrape['Date added'] = pd.to_datetime(df_scrape['Date added'], errors='raise')
# The live table carries a small number of blank CIK cells. Coercing to a nullable
# integer records them as missing rather than crashing the ingestion, which is the
# behaviour a raw-data layer needs: capture what the source actually published.
df_scrape['CIK'] = pd.to_numeric(df_scrape['CIK'], errors='coerce').astype('Int64')
missing_cik = int(df_scrape['CIK'].isna().sum())

required_scrape = ['Symbol', 'Security', 'GICS Sector', 'Date added', 'CIK']
scrape_checks = {
    "rows_present": not df_scrape.empty,
    "symbols_present": df_scrape["Symbol"].ne("").all(),
    "symbols_unique": not df_scrape["Symbol"].duplicated().any(),
    "dates_parsed": pd.api.types.is_datetime64_any_dtype(df_scrape["Date added"]),
    "cik_is_integer": pd.api.types.is_integer_dtype(df_scrape["CIK"]),
    "cik_positive_where_present": df_scrape["CIK"].dropna().gt(0).all(),
    "cik_mostly_present": missing_cik <= 5,
}
if not SCRAPE_USED_FALLBACK:
    scrape_checks["live_table_has_at_least_500_rows"] = len(df_scrape) >= 500
assert all(scrape_checks.values()), scrape_checks

print("Blank CIK cells recorded as missing:", missing_cik)
print("Scrape source used:", "inline fallback" if SCRAPE_USED_FALLBACK else "live Wikipedia page")
print("Basic rules:", scrape_checks)
print("Typed columns:", {column: str(df_scrape[column].dtype) for column in ['Date added', 'CIK']})
v_scrape = validate(df_scrape, required_scrape); v_scrape


Blank CIK cells recorded as missing: 1
Scrape source used: live Wikipedia page
Basic rules: {'rows_present': True, 'symbols_present': np.True_, 'symbols_unique': True, 'dates_parsed': True, 'cik_is_integer': True, 'cik_positive_where_present': np.True_, 'cik_mostly_present': True, 'live_table_has_at_least_500_rows': True}
Typed columns: {'Date added': 'datetime64[ns]', 'CIK': 'Int64'}


{'missing': [], 'shape': (503, 8), 'na_total': 1}

In [8]:
_ = save_csv(df_scrape.sort_values('Symbol'), prefix='scrape', site='wikipedia', table='sp500-constituents')


Saved data/raw/scrape_site-wikipedia_table-sp500-constituents_20260818-194018.csv


## Reproducibility and validation record

### Data sources and request parameters

- **Alpha Vantage:** `https://www.alphavantage.co/query`. Parameters are `function=TIME_SERIES_DAILY`, `symbol=SPY`, `outputsize=compact`, and an API key loaded from `.env`. This branch was available but was not called in the committed run because no key was configured.
- **yfinance fallback:** Yahoo Finance SPY history at `https://finance.yahoo.com/quote/SPY/history/`. The starter requested `period=3mo`, `interval=1d`, `auto_adjust=False`, and `multi_level_index=False`. This branch produced the committed API snapshot.
- **Wikipedia scrape:** `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`. The request used a 30-second timeout and `User-Agent: AFE-Homework/1.0`. The parser selected a table by the `wikitable` class and used the first matching table, which is the constituent table.

### Validation logic

- The starter `validate()` helper reports missing required columns, table shape, and total missing values for both dataframes.
- The API checks require rows, parsed and ascending unique dates, floating-point closes, and strictly positive closes.
- The scrape checks require rows, present and unique ticker symbols, parsed dates, positive CIK values where a CIK is published, and at least 500 rows for a live response. The live table carries a small number of blank CIK cells, so CIK is stored as a nullable integer and the blank count is reported rather than silently dropped or treated as zero.
- Both save steps use the starter `save_csv()` helper, which writes relative to `data/raw/` with source metadata and a retrieval timestamp in the filename.


### Confirm `.env` is ignored

The following cell asks Git to show the matching ignore rule and separately confirms that `.env` is not tracked. It does not print the file's contents.


In [9]:
ignore_check = subprocess.run(
    ["git", "check-ignore", "-v", ".env"],
    capture_output=True,
    text=True,
    check=False,
)
print("git check-ignore -v .env")
print(ignore_check.stdout.strip())
assert ignore_check.returncode == 0, ignore_check.stderr

tracked_check = subprocess.run(
    ["git", "ls-files", "--error-unmatch", ".env"],
    capture_output=True,
    text=True,
    check=False,
)
assert tracked_check.returncode != 0, ".env must not be tracked"
print("Tracked by Git?", False)


git check-ignore -v .env
.gitignore:5:.env	.env
Tracked by Git? False


## Assumptions & risks

- **Provider revision:** historical closes may change after corporate-action corrections, so this timestamped CSV records the values observed during this run.
- **Rate limits and availability:** Alpha Vantage can return an HTTP 200 response without a time series when its free cap is reached, and yfinance availability can also change.
- **Page-structure drift:** Wikipedia may rename columns or change the `wikitable` structure, which could break the scraper even when the page remains available.
- **Point-in-time snapshot:** the scraped constituent list records membership only when retrieved. It must not be treated as a historical membership table or used without considering survivorship bias.

## Connection to the semester project

The API workflow mirrors the project's provider fallback, typed market schema, validation-first handling, and timestamped raw snapshots. SPY close history is an upstream input to the monitor's returns and next-five-session realized-volatility target. The Wikipedia table demonstrates a second ingestion method but is not presented as a model input for the current risk monitor.
